# 00 - NHANES dataset assembly

Objective:
- load the core NHANES modules
- inspect and select candidate variables
- rename variables into readable English names
- merge all modules by `SEQN`
- build a clean and controlled master dataset (v1)

## 1. Setup

This section defines the project paths and imports the required libraries for dataset assembly.

In [1]:
from pathlib import Path
import pandas as pd

print(pd.__version__)

3.0.1


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "nhanes"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(PROJECT_ROOT)
print(RAW_DIR)
print(PROCESSED_DIR)

F:\DOCUMENTOS\CIENCIA DE DATOS\portfolio-data-science\morpheus-sleep-health-ml
F:\DOCUMENTOS\CIENCIA DE DATOS\portfolio-data-science\morpheus-sleep-health-ml\data\raw\nhanes
F:\DOCUMENTOS\CIENCIA DE DATOS\portfolio-data-science\morpheus-sleep-health-ml\data\processed


## 2. Load raw NHANES modules

This section loads the selected NHANES XPT files that will be used to build the first version of the sleep health master dataset.

In [3]:
def load_xpt(filename: str) -> pd.DataFrame:
    path = RAW_DIR / filename
    return pd.read_sas(path)

In [4]:
slq = load_xpt("P_SLQ.xpt")
demo = load_xpt("P_DEMO.xpt")
bmx = load_xpt("P_BMX.xpt")
bpxo = load_xpt("P_BPXO.xpt")
smq = load_xpt("P_SMQ.xpt")
alq = load_xpt("P_ALQ.xpt")
paq = load_xpt("P_PAQ.xpt")
dpq = load_xpt("P_DPQ.xpt")

## 3. Initial inspection

This section checks the structure of each module, confirms the presence of the participant identifier (`SEQN`), and reviews the available variables before selection.

In [5]:
datasets = {
    "slq": slq,
    "demo": demo,
    "bmx": bmx,
    "bpxo": bpxo,
    "smq": smq,
    "alq": alq,
    "paq": paq,
    "dpq": dpq,
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("shape:", df.shape)
    print("SEQN in columns:", "SEQN" in df.columns)
    print("first columns:", list(df.columns[:15]))


SLQ
shape: (10195, 11)
SEQN in columns: True
first columns: ['SEQN', 'SLQ300', 'SLQ310', 'SLD012', 'SLQ320', 'SLQ330', 'SLD013', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']

DEMO
shape: (15560, 29)
SEQN in columns: True
first columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMDBORN4', 'DMDYRUSZ', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'SIALANG']

BMX
shape: (14300, 22)
SEQN in columns: True
first columns: ['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML']

BPXO
shape: (11656, 12)
SEQN in columns: True
first columns: ['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']

SMQ
shape: (11137, 16)
SEQN in columns: True
first columns: ['SEQN', 'SMQ020', 'SMD030', 'SMQ040', 'SMQ050Q', 'SMQ050U', 'SMD057', 'SMQ078', 'SMD641', 'SMD650', 'SMD100FL', 'SMD100

In [6]:
for name in ["slq", "bpxo", "alq", "paq"]:
    print(f"\n{name.upper()} columns:")
    print(list(datasets[name].columns))


SLQ columns:
['SEQN', 'SLQ300', 'SLQ310', 'SLD012', 'SLQ320', 'SLQ330', 'SLD013', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']

BPXO columns:
['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']

ALQ columns:
['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ270', 'ALQ280', 'ALQ290', 'ALQ151', 'ALQ170']

PAQ columns:
['SEQN', 'PAQ605', 'PAQ610', 'PAD615', 'PAQ620', 'PAQ625', 'PAD630', 'PAQ635', 'PAQ640', 'PAD645', 'PAQ650', 'PAQ655', 'PAD660', 'PAQ665', 'PAQ670', 'PAD675', 'PAD680']


## 4. Variable selection strategy

This project starts with a parsimonious first version of the dataset.

The goal is to retain:
- sleep-related variables required for target construction
- core demographic variables
- body composition variables
- repeated blood pressure measurements
- compact behavioral variables related to smoking, alcohol use, physical activity, and depression

At this stage, only variables with clear interpretability and strong conceptual relevance are kept.

In [7]:
slq_cols = [
    "SEQN",
    "SLD012",
    "SLD013",
    "SLQ030",
    "SLQ040",
    "SLQ050",
    "SLQ120",
]

demo_cols = [
    "SEQN",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDRETH3",
    "DMDEDUC2",
    "INDFMPIR",
]

bmx_cols = [
    "SEQN",
    "BMXWT",
    "BMXHT",
    "BMXBMI",
    "BMXWAIST",
]

bpxo_cols = [
    "SEQN",
    "BPXOSY1", "BPXODI1",
    "BPXOSY2", "BPXODI2",
    "BPXOSY3", "BPXODI3",
]

smq_cols = [
    "SEQN",
    "SMQ020",
    "SMQ040",
]

alq_cols = [
    "SEQN",
    "ALQ111",
    "ALQ121",
    "ALQ130",
    "ALQ142",
]

paq_cols = [
    "SEQN",
    "PAQ605",
    "PAQ610",
    "PAD615",
    "PAQ620",
    "PAQ625",
    "PAD630",
    "PAD680",
]

dpq_cols = [
    "SEQN",
    "DPQ010", "DPQ020", "DPQ030", "DPQ040",
    "DPQ050", "DPQ060", "DPQ070", "DPQ080", "DPQ090",
]

In [8]:
slq_sub = slq[slq_cols].copy()
slq_sub.head()

,SEQN,SLD012,SLD013,SLQ030,SLQ040,SLQ050,SLQ120
0,109266.0,7.5,8.0,1.000000e+00,5.397605e-79,2.0,5.397605e-79
1,109267.0,8.0,8.0,5.397605e-79,5.397605e-79,2.0,2.000000e+00
2,109268.0,8.5,8.0,5.397605e-79,5.397605e-79,2.0,1.000000e+00
3,109271.0,10.0,13.0,5.397605e-79,5.397605e-79,1.0,3.000000e+00
4,109273.0,6.5,8.0,5.397605e-79,5.397605e-79,1.0,2.000000e+00


## 5. Sleep questionnaire variables

This section selects the core sleep variables used to define the main outcome and retain a small number of interpretable sleep-related features.

The main target will later be derived from usual sleep duration on weekdays/workdays.

### Rename sleep variables

The selected NHANES sleep codes are renamed into readable English feature names in order to improve interpretability and downstream analysis.

In [9]:
slq_rename_map = {
    "SEQN": "participant_id",
    "SLD012": "sleep_hours_weekday",
    "SLD013": "sleep_hours_weekend",
    "SLQ030": "trouble_sleeping",
    "SLQ040": "told_doctor_trouble_sleeping",
    "SLQ050": "told_doctor_sleep_disorder",
    "SLQ120": "snoring_frequency",
}

slq_sub = slq_sub.rename(columns=slq_rename_map)
slq_sub.head()

,participant_id,sleep_hours_weekday,sleep_hours_weekend,trouble_sleeping,told_doctor_trouble_sleeping,told_doctor_sleep_disorder,snoring_frequency
0,109266.0,7.5,8.0,1.000000e+00,5.397605e-79,2.0,5.397605e-79
1,109267.0,8.0,8.0,5.397605e-79,5.397605e-79,2.0,2.000000e+00
2,109268.0,8.5,8.0,5.397605e-79,5.397605e-79,2.0,1.000000e+00
3,109271.0,10.0,13.0,5.397605e-79,5.397605e-79,1.0,3.000000e+00
4,109273.0,6.5,8.0,5.397605e-79,5.397605e-79,1.0,2.000000e+00


## 6. Demographic variables

This section selects the core demographic variables used in the project, including age, sex, race/ethnicity, education, and income-to-poverty ratio.

In [10]:
demo_sub = demo[demo_cols].copy()
demo_sub.head()

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,INDFMPIR
0,109263.0,1.0,2.0,6.0,NaN,4.66
1,109264.0,2.0,13.0,1.0,NaN,0.83
2,109265.0,1.0,2.0,3.0,NaN,3.06
3,109266.0,2.0,29.0,6.0,5.0,5.00
4,109267.0,2.0,21.0,2.0,4.0,5.00


In [11]:
demo_rename_map = {
    "SEQN": "participant_id",
    "RIAGENDR": "sex",
    "RIDAGEYR": "age_years",
    "RIDRETH3": "race_ethnicity",
    "DMDEDUC2": "education_level",
    "INDFMPIR": "income_to_poverty_ratio",
}

demo_sub = demo_sub.rename(columns=demo_rename_map)

In [12]:
demo_sub.head()

,participant_id,sex,age_years,race_ethnicity,education_level,income_to_poverty_ratio
0,109263.0,1.0,2.0,6.0,NaN,4.66
1,109264.0,2.0,13.0,1.0,NaN,0.83
2,109265.0,1.0,2.0,3.0,NaN,3.06
3,109266.0,2.0,29.0,6.0,5.0,5.00
4,109267.0,2.0,21.0,2.0,4.0,5.00


In [13]:
for name in ["demo", "bmx", "smq", "dpq"]:
    print(f"\n{name.upper()} columns:")
    print(list(datasets[name].columns))


DEMO columns:
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMDBORN4', 'DMDYRUSZ', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'WTINTPRP', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA', 'INDFMPIR']

BMX columns:
['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

SMQ columns:
['SEQN', 'SMQ020', 'SMD030', 'SMQ040', 'SMQ050Q', 'SMQ050U', 'SMD057', 'SMQ078', 'SMD641', 'SMD650', 'SMD100FL', 'SMD100MN', 'SMQ670', 'SMQ621', 'SMD630', 'SMAQUEX2']

DPQ columns:
['SEQN', 'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090', 'DPQ100']


## 7. Body measures variables

This section selects anthropometric variables that capture body size and adiposity, such as weight, height, BMI, and waist circumference.

In [14]:
bmx_sub = bmx[bmx_cols].copy()
bmx_sub.head()

,SEQN,BMXWT,BMXHT,BMXBMI,BMXWAIST
0,109263.0,NaN,NaN,NaN,NaN
1,109264.0,42.2,154.7,17.6,63.8
2,109265.0,12.0,89.3,15.0,41.2
3,109266.0,97.1,160.2,37.8,117.9
4,109269.0,13.6,NaN,NaN,NaN


### Rename body measures variables

The selected NHANES body measurement codes are renamed into readable English feature names for interpretability and downstream analysis.

In [15]:
bmx_rename_map = {
    "SEQN": "participant_id",
    "BMXWT": "weight_kg",
    "BMXHT": "height_cm",
    "BMXBMI": "body_mass_index",
    "BMXWAIST": "waist_circumference_cm",
}

bmx_sub = bmx_sub.rename(columns=bmx_rename_map)
bmx_sub.head()

,participant_id,weight_kg,height_cm,body_mass_index,waist_circumference_cm
0,109263.0,NaN,NaN,NaN,NaN
1,109264.0,42.2,154.7,17.6,63.8
2,109265.0,12.0,89.3,15.0,41.2
3,109266.0,97.1,160.2,37.8,117.9
4,109269.0,13.6,NaN,NaN,NaN


## 8. Blood pressure variables

This section selects repeated systolic and diastolic blood pressure measurements.

The repeated readings are retained at this stage and will later be aggregated into mean blood pressure variables.

In [16]:
bpxo_sub = bpxo[bpxo_cols].copy()
bpxo_sub.head()

,SEQN,BPXOSY1,BPXODI1,BPXOSY2,BPXODI2,BPXOSY3,BPXODI3
0,109264.0,109.0,67.0,109.0,68.0,106.0,66.0
1,109266.0,99.0,56.0,99.0,55.0,99.0,52.0
2,109270.0,123.0,73.0,124.0,77.0,127.0,70.0
3,109271.0,102.0,65.0,108.0,68.0,111.0,68.0
4,109273.0,116.0,68.0,110.0,66.0,115.0,68.0


### Rename blood pressure variables

The selected NHANES blood pressure codes are renamed into readable English feature names for interpretability and downstream analysis.

In [17]:
bpxo_rename_map = {
    "SEQN": "participant_id",
    "BPXOSY1": "systolic_bp_1",
    "BPXODI1": "diastolic_bp_1",
    "BPXOSY2": "systolic_bp_2",
    "BPXODI2": "diastolic_bp_2",
    "BPXOSY3": "systolic_bp_3",
    "BPXODI3": "diastolic_bp_3",
}

bpxo_sub = bpxo_sub.rename(columns=bpxo_rename_map)
bpxo_sub.head()

,participant_id,systolic_bp_1,diastolic_bp_1,systolic_bp_2,diastolic_bp_2,systolic_bp_3,diastolic_bp_3
0,109264.0,109.0,67.0,109.0,68.0,106.0,66.0
1,109266.0,99.0,56.0,99.0,55.0,99.0,52.0
2,109270.0,123.0,73.0,124.0,77.0,127.0,70.0
3,109271.0,102.0,65.0,108.0,68.0,111.0,68.0
4,109273.0,116.0,68.0,110.0,66.0,115.0,68.0


## 9. Smoking variables

This section selects a compact set of smoking-related variables for the first dataset version.

The objective is to capture basic smoking exposure without adding unnecessary questionnaire detail.

In [18]:
smq_sub = smq[smq_cols].copy()
smq_sub.head()

,SEQN,SMQ020,SMQ040
0,109264.0,NaN,NaN
1,109266.0,2.0,NaN
2,109267.0,2.0,NaN
3,109268.0,2.0,NaN
4,109271.0,1.0,1.0


### Rename smoking variables

The selected NHANES smoking codes are renamed into readable English feature names for interpretability and downstream analysis.

In [19]:
smq_rename_map = {
    "SEQN": "participant_id",
    "SMQ020": "smoked_100_cigarettes_lifetime",
    "SMQ040": "current_smoking_frequency",
}

smq_sub = smq_sub.rename(columns=smq_rename_map)
smq_sub.head()

,participant_id,smoked_100_cigarettes_lifetime,current_smoking_frequency
0,109264.0,NaN,NaN
1,109266.0,2.0,NaN
2,109267.0,2.0,NaN
3,109268.0,2.0,NaN
4,109271.0,1.0,1.0


## 10. Alcohol variables

This section selects a compact and interpretable subset of alcohol-related variables for the first version of the dataset.

In [20]:
alq_sub = alq[alq_cols].copy()
alq_sub.head()

,SEQN,ALQ111,ALQ121,ALQ130,ALQ142
0,109266.0,1.0,1.000000e+01,1.0,5.397605e-79
1,109271.0,1.0,5.397605e-79,NaN,NaN
2,109273.0,1.0,5.397605e-79,NaN,NaN
3,109274.0,1.0,4.000000e+00,2.0,5.000000e+00
4,109282.0,1.0,5.397605e-79,NaN,NaN


### Rename alcohol variables

The selected NHANES alcohol-related codes are renamed into readable English feature names for interpretability and downstream analysis.

In [21]:
alq_rename_map = {
    "SEQN": "participant_id",
    "ALQ111": "alcohol_lifetime_12_drinks",
    "ALQ121": "alcohol_past_year_12_drinks",
    "ALQ130": "alcohol_frequency",
    "ALQ142": "drinks_per_day",
}

alq_sub = alq_sub.rename(columns=alq_rename_map)
alq_sub.head()

,participant_id,alcohol_lifetime_12_drinks,alcohol_past_year_12_drinks,alcohol_frequency,drinks_per_day
0,109266.0,1.0,1.000000e+01,1.0,5.397605e-79
1,109271.0,1.0,5.397605e-79,NaN,NaN
2,109273.0,1.0,5.397605e-79,NaN,NaN
3,109274.0,1.0,4.000000e+00,2.0,5.000000e+00
4,109282.0,1.0,5.397605e-79,NaN,NaN


## 11. Physical activity variables

This section selects a compact set of physical activity variables related to moderate/vigorous activity and sedentary time.

The goal is to preserve behavioral signal while keeping the first version of the dataset manageable.

In [22]:
paq_sub = paq[paq_cols].copy()
paq_sub.head()

,SEQN,PAQ605,PAQ610,PAD615,PAQ620,PAQ625,PAD630,PAD680
0,109266.0,2.0,NaN,NaN,2.0,NaN,NaN,480.0
1,109267.0,2.0,NaN,NaN,2.0,NaN,NaN,540.0
2,109268.0,1.0,5.0,540.0,1.0,5.0,300.0,540.0
3,109271.0,2.0,NaN,NaN,1.0,2.0,120.0,60.0
4,109273.0,1.0,3.0,240.0,2.0,NaN,NaN,180.0


In [23]:
paq_rename_map = {
    "SEQN": "participant_id",
    "PAQ605": "vigorous_activity",
    "PAQ610": "vigorous_activity_days",
    "PAD615": "vigorous_activity_minutes",
    "PAQ620": "moderate_activity",
    "PAQ625": "moderate_activity_days",
    "PAD630": "moderate_activity_minutes",
    "PAD680": "sedentary_minutes",
}

paq_sub = paq_sub.rename(columns=paq_rename_map)
paq_sub.head()

,participant_id,vigorous_activity,vigorous_activity_days,vigorous_activity_minutes,moderate_activity,moderate_activity_days,moderate_activity_minutes,sedentary_minutes
0,109266.0,2.0,NaN,NaN,2.0,NaN,NaN,480.0
1,109267.0,2.0,NaN,NaN,2.0,NaN,NaN,540.0
2,109268.0,1.0,5.0,540.0,1.0,5.0,300.0,540.0
3,109271.0,2.0,NaN,NaN,1.0,2.0,120.0,60.0
4,109273.0,1.0,3.0,240.0,2.0,NaN,NaN,180.0


## 12. Depression variables

This section keeps the PHQ-9 questionnaire items in order to preserve flexibility for later score construction and mental health analysis.

In [24]:
dpq_sub = dpq[dpq_cols].copy()
dpq_sub.head()

,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090
0,109266.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
1,109271.0,2.000000e+00,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,2.000000e+00,5.397605e-79,5.397605e-79
2,109273.0,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,5.397605e-79
3,109274.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
4,109282.0,5.397605e-79,1.000000e+00,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,3.000000e+00,5.397605e-79


In [25]:
dpq_rename_map = {
    "SEQN": "participant_id",
    "DPQ010": "phq_little_interest",
    "DPQ020": "phq_feeling_down",
    "DPQ030": "phq_sleep_problems",
    "DPQ040": "phq_low_energy",
    "DPQ050": "phq_appetite_problems",
    "DPQ060": "phq_feeling_bad_about_self",
    "DPQ070": "phq_trouble_concentrating",
    "DPQ080": "phq_slow_or_restless",
    "DPQ090": "phq_self_harm_thoughts",
}

dpq_sub = dpq_sub.rename(columns=dpq_rename_map)
dpq_sub.head()

,participant_id,phq_little_interest,phq_feeling_down,phq_sleep_problems,phq_low_energy,phq_appetite_problems,phq_feeling_bad_about_self,phq_trouble_concentrating,phq_slow_or_restless,phq_self_harm_thoughts
0,109266.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
1,109271.0,2.000000e+00,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,2.000000e+00,5.397605e-79,5.397605e-79
2,109273.0,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,5.397605e-79
3,109274.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
4,109282.0,5.397605e-79,1.000000e+00,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,3.000000e+00,5.397605e-79


## 13. Build module subsets overview

At this point, each NHANES module has been reduced to a compact and interpretable subset of variables.

Before merging, a quick overview is used to confirm that all module subsets were created correctly and remain aligned around the participant identifier.

In [26]:
subsets = {
    "slq_sub": slq_sub,
    "demo_sub": demo_sub,
    "bmx_sub": bmx_sub,
    "bpxo_sub": bpxo_sub,
    "smq_sub": smq_sub,
    "alq_sub": alq_sub,
    "paq_sub": paq_sub,
    "dpq_sub": dpq_sub,
}

for name, df in subsets.items():
    print(f"{name}: {df.shape}")

slq_sub: (10195, 7)
demo_sub: (15560, 6)
bmx_sub: (14300, 5)
bpxo_sub: (11656, 7)
smq_sub: (11137, 3)
alq_sub: (8965, 5)
paq_sub: (9693, 8)
dpq_sub: (8965, 10)


In [27]:
for name, df in subsets.items():
    print(f"{name}: {'participant_id' in df.columns}")

slq_sub: True
demo_sub: True
bmx_sub: True
bpxo_sub: True
smq_sub: True
alq_sub: True
paq_sub: True
dpq_sub: True


## 14. Merge all modules into a master dataset

This section merges all selected NHANES module subsets using `participant_id` as the common key.

The sleep module is used as the base table, and all additional modules are merged with left joins in order to preserve the core sleep cohort.

In [28]:
master = slq_sub.copy()

for df in [demo_sub, bmx_sub, bpxo_sub, smq_sub, alq_sub, paq_sub, dpq_sub]:
    master = master.merge(df, on="participant_id", how="left")

master.shape

(10195, 44)

In [29]:
master.head()

,participant_id,sleep_hours_weekday,sleep_hours_weekend,trouble_sleeping,told_doctor_trouble_sleeping,told_doctor_sleep_disorder,snoring_frequency,sex,age_years,race_ethnicity,...,sedentary_minutes,phq_little_interest,phq_feeling_down,phq_sleep_problems,phq_low_energy,phq_appetite_problems,phq_feeling_bad_about_self,phq_trouble_concentrating,phq_slow_or_restless,phq_self_harm_thoughts
0,109266.0,7.5,8.0,1.000000e+00,5.397605e-79,2.0,5.397605e-79,2.0,29.0,6.0,...,480.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
1,109267.0,8.0,8.0,5.397605e-79,5.397605e-79,2.0,2.000000e+00,2.0,21.0,2.0,...,540.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,109268.0,8.5,8.0,5.397605e-79,5.397605e-79,2.0,1.000000e+00,2.0,18.0,3.0,...,540.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,109271.0,10.0,13.0,5.397605e-79,5.397605e-79,1.0,3.000000e+00,1.0,49.0,3.0,...,60.0,2.000000e+00,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,2.000000e+00,5.397605e-79,5.397605e-79
4,109273.0,6.5,8.0,5.397605e-79,5.397605e-79,1.0,2.000000e+00,1.0,36.0,3.0,...,180.0,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,5.397605e-79


## 15. Post-merge quality checks

This section checks the shape of the merged dataset, the uniqueness of the participant identifier, and the initial missing-value profile.

In [30]:
master["participant_id"].duplicated().sum()

np.int64(0)

In [31]:
master.isna().mean().sort_values(ascending=False).head(20)

vigorous_activity_minutes      0.764394
vigorous_activity_days         0.762825
current_smoking_frequency      0.618538
moderate_activity_minutes      0.591564
moderate_activity_days         0.589308
drinks_per_day                 0.424914
alcohol_frequency              0.424914
alcohol_past_year_12_drinks    0.264051
phq_self_harm_thoughts         0.185679
phq_slow_or_restless           0.185483
phq_feeling_bad_about_self     0.185483
phq_trouble_concentrating      0.185483
phq_appetite_problems          0.185385
phq_low_energy                 0.185385
phq_sleep_problems             0.185287
phq_feeling_down               0.185287
phq_little_interest            0.185091
alcohol_lifetime_12_drinks     0.179009
systolic_bp_3                  0.173713
diastolic_bp_3                 0.173713
dtype: float64

In [32]:
master.shape

(10195, 44)

## 16. Initial design decisions

This section documents the initial variable selection logic used to build the first version of the NHANES sleep master dataset.

The purpose is to make the early design choices explicit, reproducible, and interpretable before moving into target construction and downstream preprocessing.

### Selection philosophy

The first dataset version was intentionally designed to remain compact, interpretable, and analytically coherent.

Variables were selected according to the following principles:
- direct or plausible relevance to sleep duration and sleep health
- strong interpretability for EDA and modeling
- manageable complexity for a first project version
- avoidance of excessive expansion into low-yield or highly specific questionnaire detail
- preference for variables that support a clear portfolio narrative

The resulting V1 combines:
- sleep-related variables
- demographic variables
- anthropometric variables
- repeated blood pressure measurements
- compact smoking, alcohol, physical activity, and depression variables

### Interpreting NHANES variable codes

NHANES variables are distributed as coded identifiers rather than human-readable feature names.

Examples:
- `SEQN`: participant identifier
- `SLD012`: usual sleep hours on weekdays/workdays
- `BMXBMI`: body mass index
- `SMQ040`: current smoking frequency

For this reason, selected variables were renamed into short, readable English names during assembly.

These names are intended to function as concise, project-facing aliases. They are designed to remain faithful to the original NHANES meaning, but they do not aim to reproduce the exact codebook wording verbatim.

This improves:
- readability
- traceability
- EDA quality
- modeling clarity
- portfolio presentation

###  Omitted variables by module

A number of original NHANES variables were intentionally excluded from the first dataset version.

- **Sleep (`SLQ`)**: `SLQ300`, `SLQ310`, `SLQ320`, `SLQ330`
  - omitted to avoid early expansion into more specific or less immediately interpretable sleep detail

- **Demographics (`DEMO`)**: survey-design variables, language/proxy/interpreter variables, and several contextual demographic fields
  - omitted because they were either technical survey variables or not essential for the first compact version

- **Body measures (`BMX`)**: technical status fields and several secondary anthropometric measures
  - omitted to keep the anthropometric block centered on weight, height, BMI, and waist circumference

- **Blood pressure (`BPXO`)**: arm, cuff-size, and pulse variables
  - omitted because the first version focuses on repeated systolic and diastolic pressure only

- **Smoking (`SMQ`)**: quantity/timing/dependence detail and several additional smoking fields
  - omitted to preserve a compact first smoking block based on lifetime exposure and current frequency

- **Alcohol (`ALQ`)**: additional alcohol-use detail beyond exposure, frequency, and average intake
  - omitted to keep the alcohol block compact and interpretable

- **Physical activity (`PAQ`)**: additional activity-detail variables beyond vigorous activity, moderate activity, and sedentary time
  - omitted to avoid overcomplicating the first behavioral representation

- **Depression (`DPQ`)**: `DPQ100`
  - omitted to keep the PHQ-9 core block compact in the first version

###  Future expansion logic

The current V1 is not intended to capture every available NHANES variable related to sleep.

Instead, it provides a strong and interpretable baseline dataset.

Potential V2 expansions may later include:
- more detailed smoking variables related to dependence or timing
- richer alcohol-use detail
- additional physical activity variables
- selected laboratory biomarkers
- selected dietary or metabolic variables

These were intentionally left out of V1 to avoid unnecessary complexity during the first assembly stage.

### Variables flagged for later modeling review

Some variables are retained in the assembled dataset because they are relevant to sleep health, but they may later require closer review before modeling.

In particular, variables that directly reflect sleep complaints, sleep diagnoses, or closely related depressive symptoms may introduce conceptual redundancy or leakage with the final target.

These variables are intentionally preserved at the assembly stage and will be audited during EDA before defining the final modeling feature set.

## 17. Recode NHANES special missing values

NHANES variables often use special numeric codes to represent missing, non-applicable, refused, or unknown responses.

Before creating the main target and aggregated clinical variables, these special values must be converted to proper missing values (`NaN`) in order to avoid contaminating downstream features.

This step is limited to structural cleaning required for valid assembly outputs.

In [33]:
special_missing_values = {
    7, 9,
    77, 99,
    777, 999,
    7777, 9999,
    77777, 99999,
}

columns_to_recode = [

    # demographics
    "education_level",

    # sleep
    "sleep_hours_weekday",
    "sleep_hours_weekend",
    "trouble_sleeping",
    "told_doctor_trouble_sleeping",
    "told_doctor_sleep_disorder",
    "snoring_frequency",

    # blood pressure
    "systolic_bp_1", "diastolic_bp_1",
    "systolic_bp_2", "diastolic_bp_2",
    "systolic_bp_3", "diastolic_bp_3",

    # smoking
    "smoked_100_cigarettes_lifetime",
    "current_smoking_frequency",

    # alcohol
    "alcohol_lifetime_12_drinks",
    "alcohol_past_year_12_drinks",
    "alcohol_frequency",
    "drinks_per_day",

    # physical activity
    "vigorous_activity",
    "vigorous_activity_days",
    "vigorous_activity_minutes",
    "moderate_activity",
    "moderate_activity_days",
    "moderate_activity_minutes",
    "sedentary_minutes",

    # depression
    "phq_little_interest",
    "phq_feeling_down",
    "phq_sleep_problems",
    "phq_low_energy",
    "phq_appetite_problems",
    "phq_feeling_bad_about_self",
    "phq_trouble_concentrating",
    "phq_slow_or_restless",
    "phq_self_harm_thoughts",
]


In [34]:
for col in columns_to_recode:
    master[col] = master[col].replace(special_missing_values, pd.NA)

In [35]:
for col in ["sex", "race_ethnicity", "education_level", "income_to_poverty_ratio"]:
    print(f"\n{col}")
    print(master[col].value_counts(dropna=False).sort_index())


sex
sex
1.0    4969
2.0    5226
Name: count, dtype: int64

race_ethnicity
race_ethnicity
1.0    1208
2.0    1025
3.0    3524
4.0    2685
6.0    1226
7.0     527
Name: count, dtype: int64

education_level
education_level
1.0      719
2.0     1041
3.0     2225
4.0     2975
5.0     2257
NaN      963
<NA>      15
Name: count, dtype: int64

income_to_poverty_ratio
income_to_poverty_ratio
5.397605e-79      79
1.000000e-02       8
2.000000e-02       9
3.000000e-02      10
4.000000e-02      21
                ... 
4.960000e+00       2
4.970000e+00       8
4.980000e+00      10
5.000000e+00    1570
NaN             1539
Name: count, Length: 473, dtype: int64


In [36]:
master["income_to_poverty_ratio"].describe()

count    8.656000e+03
mean     2.558194e+00
std      1.632875e+00
min      5.397605e-79
25%      1.160000e+00
50%      2.160000e+00
75%      4.140000e+00
max      5.000000e+00
Name: income_to_poverty_ratio, dtype: float64

In [37]:
master["income_to_poverty_ratio"].sort_values().head(20)

10173    5.397605e-79
8337     5.397605e-79
1390     5.397605e-79
1459     5.397605e-79
2581     5.397605e-79
1521     5.397605e-79
7514     5.397605e-79
5807     5.397605e-79
8279     5.397605e-79
1338     5.397605e-79
1337     5.397605e-79
729      5.397605e-79
716      5.397605e-79
2686     5.397605e-79
2479     5.397605e-79
5968     5.397605e-79
2430     5.397605e-79
2857     5.397605e-79
6013     5.397605e-79
2844     5.397605e-79
Name: income_to_poverty_ratio, dtype: float64

In [38]:
master["income_to_poverty_ratio"].sort_values(ascending=False).head(20)

6615     5.0
6623     5.0
0        5.0
10186    5.0
10187    5.0
10190    5.0
16       5.0
8855     5.0
8857     5.0
8859     5.0
8863     5.0
2533     5.0
2544     5.0
2545     5.0
2550     5.0
2557     5.0
2559     5.0
2565     5.0
8852     5.0
85       5.0
Name: income_to_poverty_ratio, dtype: float64

In [39]:
(master["income_to_poverty_ratio"] < 1e-10).sum()

np.int64(79)

In [40]:
master.loc[
    master["income_to_poverty_ratio"] < 1e-10,
    "income_to_poverty_ratio"
] = pd.NA

In [41]:
master["income_to_poverty_ratio"].describe()

count    8577.000000
mean        2.581757
std         1.621728
min         0.010000
25%         1.180000
50%         2.190000
75%         4.160000
max         5.000000
Name: income_to_poverty_ratio, dtype: float64

In [42]:
master["income_to_poverty_ratio"].sort_values().head(20)

2539    0.01
3559    0.01
1382    0.01
3054    0.01
6738    0.01
8161    0.01
1112    0.01
4302    0.01
5489    0.02
6831    0.02
3786    0.02
510     0.02
3756    0.02
6804    0.02
9037    0.02
3383    0.02
15      0.02
7559    0.03
9063    0.03
4647    0.03
Name: income_to_poverty_ratio, dtype: float64

A small number of `income_to_poverty_ratio` values appeared as extremely small numerical artifacts. These cases were counted, reviewed, and then recoded as missing values because they are not interpretable as plausible poverty-to-income ratios.

In [43]:
master[columns_to_recode].head()

,education_level,sleep_hours_weekday,sleep_hours_weekend,trouble_sleeping,told_doctor_trouble_sleeping,told_doctor_sleep_disorder,snoring_frequency,systolic_bp_1,diastolic_bp_1,systolic_bp_2,...,sedentary_minutes,phq_little_interest,phq_feeling_down,phq_sleep_problems,phq_low_energy,phq_appetite_problems,phq_feeling_bad_about_self,phq_trouble_concentrating,phq_slow_or_restless,phq_self_harm_thoughts
0,5.0,7.5,8.0,1.0,0.0,2.0,0.0,<NA>,56.0,<NA>,...,480.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.0,8.0,8.0,0.0,0.0,2.0,2.0,NaN,NaN,NaN,...,540.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,8.5,8.0,0.0,0.0,2.0,1.0,NaN,NaN,NaN,...,540.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,10.0,13.0,0.0,0.0,1.0,3.0,102.0,65.0,108.0,...,60.0,2.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,4.0,6.5,8.0,0.0,0.0,1.0,2.0,116.0,68.0,110.0,...,180.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,0.0


In [44]:
master[columns_to_recode].isna().mean().sort_values(ascending=False).head(15)

vigorous_activity_days         0.789112
vigorous_activity_minutes      0.765179
moderate_activity_days         0.644237
current_smoking_frequency      0.618538
moderate_activity_minutes      0.593232
drinks_per_day                 0.493085
alcohol_frequency              0.431486
alcohol_past_year_12_drinks    0.398333
sleep_hours_weekend            0.302697
sleep_hours_weekday            0.285140
diastolic_bp_1                 0.200294
diastolic_bp_2                 0.200000
diastolic_bp_3                 0.199411
phq_feeling_bad_about_self     0.186366
phq_self_harm_thoughts         0.186268
dtype: float64

## 18. Create the main sleep target

This section defines the primary modeling target for the project.

The target is based on usual sleep duration on weekdays/workdays, and classifies participants as short sleepers when reported sleep is below 7 hours.

In [45]:
master["short_sleep"] = pd.Series(pd.NA, index=master.index, dtype="Int64")

mask_notna = master["sleep_hours_weekday"].notna()

master.loc[mask_notna, "short_sleep"] = (
    master.loc[mask_notna, "sleep_hours_weekday"] < 7
).astype("Int64")

In [46]:
master[["sleep_hours_weekday", "short_sleep"]].head()

,sleep_hours_weekday,short_sleep
0,7.5,0
1,8.0,0
2,8.5,0
3,10.0,0
4,6.5,1


In [47]:
master["short_sleep"].value_counts(dropna=False)

short_sleep
0       4804
<NA>    2907
1       2484
Name: count, dtype: Int64

In [48]:
master["short_sleep"].value_counts(normalize=True, dropna=False)

short_sleep
0       0.471211
<NA>     0.28514
1       0.243649
Name: proportion, dtype: Float64

In [49]:
master["sleep_hours_weekday"].isna().mean(), master["short_sleep"].isna().mean()

(np.float64(0.2851397743992153), np.float64(0.2851397743992153))

## 19. Aggregate repeated blood pressure measurements

This section creates mean systolic and diastolic blood pressure variables from the repeated measurements available in the NHANES blood pressure module.

Using mean values provides a more stable representation than relying on a single reading.

In [50]:
master["systolic_bp_mean"] = master[
    ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
].mean(axis=1)

master["diastolic_bp_mean"] = master[
    ["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]
].mean(axis=1)


In [51]:
master[
    [
        "systolic_bp_1", "systolic_bp_2", "systolic_bp_3",
        "systolic_bp_mean",
        "diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3",
        "diastolic_bp_mean",
    ]
].head()

,systolic_bp_1,systolic_bp_2,systolic_bp_3,systolic_bp_mean,diastolic_bp_1,diastolic_bp_2,diastolic_bp_3,diastolic_bp_mean
0,<NA>,<NA>,<NA>,NaN,56.0,55.0,52.0,54.333333
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,102.0,108.0,111.0,107.0,65.0,68.0,68.0,67.0
4,116.0,110.0,115.0,113.666667,68.0,66.0,68.0,67.333333


In [52]:
master[["systolic_bp_mean", "diastolic_bp_mean"]].isna().mean()

systolic_bp_mean     0.170574
diastolic_bp_mean    0.171162
dtype: float64

In [53]:
master[
    ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
].head(10)

,systolic_bp_1,systolic_bp_2,systolic_bp_3
0,<NA>,<NA>,<NA>
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,102.0,108.0,111.0
4,116.0,110.0,115.0
5,138.0,132.0,132.0
6,100.0,103.0,101.0
7,141.0,137.0,140.0
8,NaN,NaN,NaN
9,NaN,NaN,NaN


In [54]:
master["short_sleep"].value_counts(dropna=False)

short_sleep
0       4804
<NA>    2907
1       2484
Name: count, dtype: Int64

### Additional artifact cleanup

A tiny floating-point artifact was detected in several variables during EDA. These values do not represent valid responses and are recoded to missing before saving the assembled dataset.

In [55]:
artifact_cols = [
    "told_doctor_trouble_sleeping",
    "drinks_per_day",
    "alcohol_past_year_12_drinks",
    "trouble_sleeping",
    "snoring_frequency",
    "phq_little_interest",
    "phq_feeling_down",
    "phq_sleep_problems",
    "phq_low_energy",
    "phq_appetite_problems",
    "phq_feeling_bad_about_self",
    "phq_trouble_concentrating",
    "phq_slow_or_restless",
    "phq_self_harm_thoughts",
]

for col in artifact_cols:
    master.loc[(master[col] != 0) & (master[col].abs() < 1e-20), col] = pd.NA

## 20. Save the assembled master dataset

This section saves the assembled dataset after variable selection, renaming, merging, target creation, and basic structural feature construction.

This file is an assembly-stage output, not the final modeling dataset.

In [56]:
output_path = PROCESSED_DIR / "nhanes_sleep_master_v1.csv"
master.to_csv(output_path, index=False)

output_path

WindowsPath('F:/DOCUMENTOS/CIENCIA DE DATOS/portfolio-data-science/morpheus-sleep-health-ml/data/processed/nhanes_sleep_master_v1.csv')